In [0]:
%run "../utils/00_config"

In [0]:
# Read all execution manifests from Bronze Volume
manifest_path = (
    f"{RAW_PATH}"
    "execution_date=*/execution_id=*/manifest.json"
)

execution_manifest = (
    spark.read
    .option("multiLine", "true")
    .json(manifest_path)
    .withColumn("started_at", F.to_timestamp("started_at"))
    .withColumn("finished_at", F.to_timestamp("finished_at"))
    .withColumn(
        "execution_duration_seconds",
        F.unix_timestamp("finished_at")
        - F.unix_timestamp("started_at")
    )
    .withColumn(
        "source_file",
        F.col("_metadata.file_path")
    )
)

display(execution_manifest)

In [0]:
# Validate execution manifest integrity
validation = execution_manifest.agg(
    F.count("*").alias("total_records"),
    F.countDistinct("execution_id").alias("unique_executions"),
    F.sum(
        F.when(F.col("execution_id").isNull(), 1).otherwise(0)
    ).alias("null_execution_ids"),
    F.sum(
        F.when(
            F.col("started_at").isNull()
            | F.col("finished_at").isNull(),
            1
        ).otherwise(0)
    ).alias("null_timestamps"),
    F.sum(
        F.when(F.col("status") == "success", 1).otherwise(0)
    ).alias("successful_executions")
)

display(validation)

In [0]:
from delta.tables import DeltaTable

TARGET_TABLE = f"{CATALOGS['bronze']}.warframe_market.execution_manifest"

if spark.catalog.tableExists(TARGET_TABLE):

    target = DeltaTable.forName(spark, TARGET_TABLE)

    (
        target.alias("target")
        .merge(
            execution_manifest.alias("source"),
            "target.execution_id = source.execution_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

else:

    (
        execution_manifest.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(TARGET_TABLE)
    )

print(f"Table successfully materialized: {TARGET_TABLE}")